# Fine-tune Falcon3-3B-Instruct with Unsloth

This notebook trains a 16-bit LoRA adapter over 5 epochs. The effective batch size is 32; samples exceeding 32K tokens are dropped rather than truncated.

## 1. Install dependencies

This cell installs Unsloth and the minimum libraries needed for datasets, PEFT, training, and plotting. After the first install, restart the kernel if the environment requires it.

In [ ]:
%pip install -q -U unsloth datasets transformers trl peft accelerate pandas matplotlib jupyter ipykernel ipywidgets jupyterlab_widgets widgetsnbextension

## 2. Configuration

This cell pins the model, context length, batch size, epoch count, and paths relative to the repo root. The notebook stops early if the pickle file or CUDA is unavailable.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import pickle
import re
import shutil
import subprocess

import torch

MODEL_NAME = "tiiuae/Falcon3-3B-Instruct"
MAX_SEQ_LENGTH = 32768
VARIANT = "instruct"
DATA_PATH = Path("/workspace/fine-tuning/synthtraces_dataset.pickle")
OUTPUT_DIR = Path(f"outputs/unsloth/{VARIANT}")
FINAL_DIR = OUTPUT_DIR / "adapter-final"
GPU_LOG_PATH = OUTPUT_DIR / "gpu_usage.jsonl"

MICRO_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8
NUM_EPOCHS = 10
LEARNING_RATE = 2e-4
SEED = 3407
RESUME_FROM_LAST_CHECKPOINT = True

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File not found: {DATA_PATH}. Run Jupyter from the repo root."
    )
if not torch.cuda.is_available():
    raise RuntimeError("This Unsloth notebook requires a CUDA GPU.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
has_checkpoint = any(OUTPUT_DIR.glob("checkpoint-*"))
if GPU_LOG_PATH.exists() and not (RESUME_FROM_LAST_CHECKPOINT and has_checkpoint):
    GPU_LOG_PATH.unlink()
print({
    "model": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "effective_batch_size": MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "output_dir": str(OUTPUT_DIR),
})

## 3. Load pickle

This cell reads the payload and performs basic schema validation. Input is synthtraces_dataset.pickle; output is a list of rows in memory.

In [ ]:
with DATA_PATH.open("rb") as handle:
    payload = pickle.load(handle)

rows = payload["rows"]
if not rows:
    raise ValueError("Dataset contains no rows.")

print("payload keys:", list(payload))
print("rows:", len(rows))
print("row keys:", list(rows[0]))

## 4. GPU logger setup

The logger prefers nvidia-smi for utilization, temperature, and memory data; if the command is unavailable it falls back to PyTorch memory metrics.

In [4]:
from datetime import datetime, timezone

def gpu_snapshot(event, step=None, epoch=None):
    row = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "event": event,
        "step": step,
        "epoch": epoch,
    }
    if torch.cuda.is_available():
        device = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(device)
        row.update({
            "device": device,
            "gpu_name": props.name,
            "torch_allocated_gib": torch.cuda.memory_allocated(device) / 1024**3,
            "torch_reserved_gib": torch.cuda.memory_reserved(device) / 1024**3,
            "torch_max_allocated_gib": torch.cuda.max_memory_allocated(device) / 1024**3,
            "memory_total_gib": props.total_memory / 1024**3,
        })
        try:
            query = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,temperature.gpu,memory.used,memory.total",
                    "--format=csv,noheader,nounits",
                    f"--id={device}",
                ],
                check=True,
                capture_output=True,
                text=True,
            ).stdout.strip().splitlines()[0]
            utilization, temperature, used_mib, total_mib = [
                float(value.strip()) for value in query.split(",")
            ]
            row.update({
                "gpu_utilization_percent": utilization,
                "temperature_c": temperature,
                "nvidia_memory_used_gib": used_mib / 1024,
                "nvidia_memory_total_gib": total_mib / 1024,
            })
        except (FileNotFoundError, subprocess.CalledProcessError, IndexError, ValueError):
            row["gpu_utilization_percent"] = None
            row["temperature_c"] = None
            row["nvidia_memory_used_gib"] = row["torch_reserved_gib"]
            row["nvidia_memory_total_gib"] = row["memory_total_gib"]
    else:
        row["cuda_available"] = False
    return row

def log_gpu(event, step=None, epoch=None):
    row = gpu_snapshot(event, step, epoch)
    GPU_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with GPU_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("[GPU]", row)
    return row

## 5. Load Instruct model

Unsloth loads the model in 16-bit with unsloth_tiled_mlp=True. The Instruct variant keeps the native tokenizer and chat template; the notebook stops if the checkpoint does not provide a chat template.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
    trust_remote_code=True,
    unsloth_tiled_mlp=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_SEQ_LENGTH

if not tokenizer.chat_template:
    raise ValueError("Falcon3-Instruct tokenizer has no native chat_template.")

print("tokenizer size:", len(tokenizer))
print("chat template available:", bool(tokenizer.chat_template))
log_gpu("model_loaded")

## 6. Define OpenAI-format preprocessing

The Instruct path retains messages, tools, tool_calls, and the tool role per the OpenAI schema. This cell only normalizes absolute paths before calling the native chat template.

In [6]:
def parse_json_if_needed(value, default):
    if value is None:
        return default
    return json.loads(value) if isinstance(value, str) else value

def normalize_path_text(value, cwd):
    if not isinstance(value, str):
        return value
    if cwd:
        value = value.replace(cwd + "/", "").replace(cwd, ".")
    return re.sub(
        r"/(?:Users|home)/[^/]+/(?:Desktop/)?synthtraces/repos/[^/\s\"']+",
        "$REPO_ROOT",
        value,
    )

def normalize_nested(value, cwd):
    if isinstance(value, dict):
        return {key: normalize_nested(item, cwd) for key, item in value.items()}
    if isinstance(value, list):
        return [normalize_nested(item, cwd) for item in value]
    return normalize_path_text(value, cwd)

def normalize_openai_row(row):
    metadata = parse_json_if_needed(row.get("metadata"), {})
    cwd = metadata.get("cwd", "")
    raw_messages = parse_json_if_needed(row.get("messages"), [])
    tools = normalize_nested(parse_json_if_needed(row.get("tools"), []), cwd)
    messages = []

    for raw in raw_messages:
        role = raw.get("role")
        if role not in {"system", "user", "assistant", "tool"}:
            continue
        message = {"role": role, "content": normalize_path_text(raw.get("content", ""), cwd)}
        if raw.get("tool_calls"):
            message["tool_calls"] = normalize_nested(raw["tool_calls"], cwd)
        if role == "tool":
            message["tool_call_id"] = raw.get("tool_call_id", "")
            if raw.get("name"):
                message["name"] = raw["name"]
        if message["content"] or message.get("tool_calls"):
            messages.append(message)

    if not any(message["role"] == "assistant" for message in messages):
        return None
    if not any(message.get("tool_calls") for message in messages):
        return None
    if not tools:
        return None

    input_ids = tokenizer.apply_chat_template(
        messages,
        tools=tools or None,
        tokenize=True,
        add_generation_prompt=False,
    )
    if isinstance(input_ids, torch.Tensor):
        input_ids = input_ids.tolist()
    if not input_ids:
        raise ValueError("Chat template returned an empty sample.")
    codebase = cwd.rstrip("/").split("/")[-1] if cwd else "unknown"
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": input_ids.copy(),
        "length": len(input_ids),
        "session_id": row.get("session_id"),
        "codebase": codebase,
        "messages": messages,
        "tools": tools,
    }

## 7. Render, length-filter, and split

Each trace is rendered then tokenized without truncation. Samples longer than 32768 tokens are dropped; codebases are deterministically hashed with MD5 into an 80/10/10 split to prevent leakage.

In [ ]:
from collections import Counter
from datasets import Dataset
from tqdm.auto import tqdm

processed = []
bad_rows = []
too_long = []

for index, row in enumerate(tqdm(rows, desc="Render and tokenize")):
    try:
        example = normalize_openai_row(row)
        if example is None:
            bad_rows.append((index, "missing_assistant_or_tool_call"))
            continue
        if example["length"] > MAX_SEQ_LENGTH:
            too_long.append(example["length"])
            continue
        processed.append(example)
    except Exception as exc:
        bad_rows.append((index, f"{type(exc).__name__}: {exc}"))

if not processed:
    raise ValueError("No valid samples remain after preprocessing.")

def split_by_codebase(codebase):
    bucket = int(hashlib.md5((codebase or "unknown").encode()).hexdigest(), 16) % 100
    return "train" if bucket < 80 else "validation" if bucket < 90 else "test"

split_rows = {"train": [], "validation": [], "test": []}
for example in processed:
    split_rows[split_by_codebase(example["codebase"])].append(example)

if not split_rows["train"] or not split_rows["validation"]:
    raise ValueError("Train/validation split is empty; check codebase hash logic.")

train_codebases = {row["codebase"] for row in split_rows["train"]}
validation_codebases = {row["codebase"] for row in split_rows["validation"]}
test_codebases = {row["codebase"] for row in split_rows["test"]}
if train_codebases & validation_codebases or train_codebases & test_codebases or validation_codebases & test_codebases:
    raise AssertionError("Codebase leakage detected between splits.")

manifest = sorted(
    [
        {
            "session_id": item["session_id"],
            "split": split_name,
            "length": item["length"],
        }
        for split_name, items in split_rows.items()
        for item in items
    ],
    key=lambda item: (str(item["session_id"]), item["split"], item["length"]),
)
manifest_path = Path("outputs/manifests") / f"{VARIANT}.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
if manifest_path.exists():
    existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if existing_manifest != manifest:
        raise AssertionError(
            f"Manifest mismatch at {manifest_path}; backend did not preserve the same sample/split/length."
        )
else:
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True),
        encoding="utf-8",
    )

keep_columns = {"input_ids", "attention_mask", "labels", "length"}
def to_dataset(items):
    return Dataset.from_list([
        {key: value for key, value in item.items() if key in keep_columns}
        for item in items
    ])

train_dataset = to_dataset(split_rows["train"])
eval_dataset = to_dataset(split_rows["validation"])
test_dataset = to_dataset(split_rows["test"]) if split_rows["test"] else None

print("kept:", len(processed))
print("dropped too long:", len(too_long))
print("bad rows:", len(bad_rows))
print("split sizes:", {name: len(items) for name, items in split_rows.items()})
print("length min/max:", min(item["length"] for item in processed), max(item["length"] for item in processed))
print("top codebases:", Counter(item["codebase"] for item in processed).most_common(10))
print("first bad rows:", bad_rows[:5])

## 8. Attach LoRA

This cell verifies all seven target modules are present, then attaches LoRA r=16, alpha=32 with Unsloth gradient checkpointing.

In [ ]:
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]
module_names = {name.rsplit(".", 1)[-1] for name, _ in model.named_modules()}
missing_targets = [name for name in TARGET_MODULES if name not in module_names]
if missing_targets:
    raise ValueError(f"Model is missing LoRA target modules: {missing_targets}")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=TARGET_MODULES,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    
)
model.print_trainable_parameters()


log_gpu("lora_attached")

## 9. Dynamic padding and backward smoke test

A real batch is forwarded/backwarded before full training. This cell confirms the loss is finite and LoRA layers receive gradients; CUDA OOM is converted into a clear error message.

In [ ]:
from dataclasses import dataclass

@dataclass
class DynamicCausalCollator:
    tokenizer: object
    pad_to_multiple_of: int = 8

    def __call__(self, features):
        longest = max(len(feature["input_ids"]) for feature in features)
        padded_length = (
            (longest + self.pad_to_multiple_of - 1)
            // self.pad_to_multiple_of
            * self.pad_to_multiple_of
        )
        input_ids, attention_mask, labels = [], [], []
        for feature in features:
            pad_length = padded_length - len(feature["input_ids"])
            input_ids.append(feature["input_ids"] + [self.tokenizer.pad_token_id] * pad_length)
            attention_mask.append(feature["attention_mask"] + [0] * pad_length)
            labels.append(feature["labels"] + [-100] * pad_length)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

data_collator = DynamicCausalCollator(tokenizer)
model.train()
smoke_index = max(range(len(train_dataset)), key=lambda index: train_dataset[index]["length"])
smoke_example = train_dataset[smoke_index]
smoke_batch = data_collator([smoke_example])
smoke_batch = {key: value.to(model.device) for key, value in smoke_batch.items()}

try:
    smoke_loss = model(**smoke_batch).loss
    if not torch.isfinite(smoke_loss):
        raise FloatingPointError(f"Smoke loss is not finite: {smoke_loss.item()}")
    smoke_loss.backward()
except torch.cuda.OutOfMemoryError as exc:
    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    raise RuntimeError(
        f"Sample with {smoke_example['length']} tokens exceeds VRAM. "
        "Reduce context/sample or use a larger GPU; this notebook does not truncate."
    ) from exc

lora_has_gradient = any(
    parameter.grad is not None
    for name, parameter in model.named_parameters()
    if "lora_" in name and parameter.requires_grad
)
if not lora_has_gradient:
    raise AssertionError("LoRA layers did not receive gradients during smoke test.")



print("smoke loss:", float(smoke_loss.detach().cpu()))
model.zero_grad(set_to_none=True)
del smoke_batch, smoke_loss
torch.cuda.empty_cache()
log_gpu("backward_smoke_test_passed")

## 10. Train for 10 epochs

The Trainer uses micro-batch 1, gradient accumulation 32, learning rate 2e-4, BF16 if the GPU supports it and FP16 otherwise. A checkpoint is saved every epoch, keeping the two most recent, and training auto-resumes from the latest checkpoint.

In [ ]:
from transformers import Trainer, TrainerCallback, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

class GpuUsageCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        print(f"[train_log] step={state.global_step} epoch={state.epoch} logs={logs}", flush=True)

        if state.global_step % 10 == 0:
            log_gpu("train_log", state.global_step, state.epoch)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        print(f"[evaluate] step={state.global_step} epoch={state.epoch} metrics={metrics}", flush=True)
        log_gpu("evaluate", state.global_step, state.epoch)

    def on_train_end(self, args, state, control, **kwargs):
        log_gpu("train_end", state.global_step, state.epoch)


compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    run_name=f"falcon3-3b-{VARIANT}-unsloth",

    per_device_train_batch_size=MICRO_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    weight_decay=0.0,

    logging_strategy="steps",
    logging_steps=5,          # log loss more frequently
    logging_first_step=True,
    
    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,

    gradient_checkpointing=False,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
    seed=SEED,

    disable_tqdm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    callbacks=[GpuUsageCallback()],
)

print("train keys:", train_dataset[0].keys())
print("eval keys:", eval_dataset[0].keys())

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.exists() else None
resume_checkpoint = last_checkpoint if RESUME_FROM_LAST_CHECKPOINT else None

log_gpu("train_begin", trainer.state.global_step, trainer.state.epoch)
print("resume checkpoint:", resume_checkpoint, flush=True)

try:
    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
except torch.cuda.OutOfMemoryError as exc:
    torch.cuda.empty_cache()
    raise RuntimeError(
        f"Training context {MAX_SEQ_LENGTH} exceeds VRAM. "
        "Data is kept intact without truncation; use a larger GPU or adjust the configuration intentionally."
    ) from exc

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

eval_metrics = trainer.evaluate()
trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)

all_metrics = {
    "train": train_result.metrics,
    "eval": eval_metrics,
}

print(all_metrics, flush=True)

## 11. Save final adapter

This cell saves the adapter safetensors, config, tokenizer, metrics, and GPU log to outputs/unsloth/instruct/adapter-final. The full model is not merged or saved.

In [ ]:
print(trainer.state.log_history[-5:])
print("best_checkpoint:", trainer.state.best_model_checkpoint)
print("best_metric:", trainer.state.best_metric)

In [ ]:
import shutil

OUTPUT_DIR = Path(f"/workspace/dungvm/outputs")
FINAL_DIR = OUTPUT_DIR / "adapter-3B-instruct-final"
GPU_LOG_PATH = OUTPUT_DIR / "gpu_usage.jsonl"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
unwrapped_model = trainer.accelerator.unwrap_model(trainer.model)
unwrapped_model.save_pretrained(FINAL_DIR, safe_serialization=True)
tokenizer.save_pretrained(FINAL_DIR)

all_metrics = {
    "train": train_result.metrics if "train_result" in globals() else {},
    "eval": eval_metrics if "eval_metrics" in globals() else {},
}

(FINAL_DIR / "metrics.json").write_text(
    json.dumps(all_metrics, ensure_ascii=False, indent=2, default=float),
    encoding="utf-8",
)

if GPU_LOG_PATH.exists() and GPU_LOG_PATH.stat().st_size > 0:
    shutil.copy2(GPU_LOG_PATH, FINAL_DIR / "gpu_usage.jsonl")
else:
    print(f"No GPU log found at: {GPU_LOG_PATH}")

required_files = [
    FINAL_DIR / "adapter_config.json",
    FINAL_DIR / "adapter_model.safetensors",
    FINAL_DIR / "tokenizer_config.json",
    FINAL_DIR / "metrics.json",
]

missing = [str(path) for path in required_files if not path.exists() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError(f"Adapter output missing or empty: {missing}")

print("Adapter saved:")
for path in sorted(FINAL_DIR.iterdir()):
    if path.is_file():
        print(path.name, path.stat().st_size, "bytes")

## 12. Reload adapter and generation smoke test

This cell frees the training model, reloads the tokenizer along with the pristine base checkpoint, resizes embeddings if needed, and then loads the adapter. A short generation confirms the saved artifacts are reusable.

In [ ]:
import gc
from peft import PeftModel
from transformers import AutoTokenizer

del trainer, unwrapped_model, model
gc.collect()
torch.cuda.empty_cache()

reload_tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
reload_model, _ = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
    trust_remote_code=True,
    unsloth_tiled_mlp=True,
)
if len(reload_tokenizer) != reload_model.get_input_embeddings().num_embeddings:
    reload_model.resize_token_embeddings(len(reload_tokenizer))
reload_model = PeftModel.from_pretrained(reload_model, FINAL_DIR)
FastLanguageModel.for_inference(reload_model)

fixture_messages = [{"role": "user", "content": "List the public API of this repository."}]
fixture_tools = [{
    "type": "function",
    "function": {
        "name": "read",
        "description": "Read a repository file",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    },
}]
generation_inputs = reload_tokenizer.apply_chat_template(
    fixture_messages,
    tools=fixture_tools,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True,
).to(reload_model.device)

with torch.inference_mode():
    generated = reload_model.generate(
        **generation_inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=reload_tokenizer.pad_token_id,
        eos_token_id=reload_tokenizer.eos_token_id,
    )
print(reload_tokenizer.decode(generated[0], skip_special_tokens=False))
log_gpu("adapter_reload_generation_passed")

## 13. Loss and GPU usage plots

The final cell reads the trainer state and GPU JSONL, displays tables, and plots loss, utilization, and memory. If training has not finished, it reports the missing files instead of generating fake data.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = pd.DataFrame(trainer_state for trainer_state in [])
state_path = OUTPUT_DIR / "trainer_state.json"
if not state_path.exists():
    checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"), key=lambda path: int(path.name.split("-")[-1]))
    if checkpoints:
        state_path = checkpoints[-1] / "trainer_state.json"

if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    history = pd.DataFrame(state.get("log_history", []))
    display(history.tail())
    loss_columns = [column for column in ["loss", "eval_loss"] if column in history]
    if loss_columns and "step" in history:
        history.plot(x="step", y=loss_columns, figsize=(10, 4), marker="o", title="Training loss")
        plt.grid(True)
        plt.show()
else:
    print("trainer_state.json not found.")

final_gpu_log = FINAL_DIR / "gpu_usage.jsonl"
if final_gpu_log.exists() and final_gpu_log.stat().st_size:
    gpu = pd.read_json(final_gpu_log, lines=True)
    display(gpu.tail())
    gpu_columns = [
        column
        for column in ["gpu_utilization_percent", "nvidia_memory_used_gib", "torch_reserved_gib"]
        if column in gpu
    ]
    if gpu_columns:
        gpu.reset_index().plot(
            x="index",
            y=gpu_columns,
            figsize=(11, 4),
            marker="o",
            title="GPU utilization and memory",
        )
        plt.grid(True)
        plt.show()

In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_DIR = Path("/workspace/models/Falcon3-3B-Instruct")
ADAPTER_DIR = Path("/workspace/models/adapter-3B-instruct-final")
MERGED_DIR = Path("/workspace/models/Falcon3-3B-Instruct-merged-lora")

print("Loading tokenizer from adapter...")
tokenizer = AutoTokenizer.from_pretrained(
    str(ADAPTER_DIR),
    local_files_only=True,
    trust_remote_code=True,
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    str(BASE_DIR),
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True,
    trust_remote_code=True,
)

print("Base vocab size:", base_model.get_input_embeddings().weight.shape[0])
print("Tokenizer vocab size:", len(tokenizer))

if base_model.get_input_embeddings().weight.shape[0] != len(tokenizer):
    print("Resizing token embeddings...")
    base_model.resize_token_embeddings(len(tokenizer))

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
    local_files_only=True,
)

print("Merging LoRA...")
model = model.merge_and_unload()

MERGED_DIR.mkdir(parents=True, exist_ok=True)

print("Saving merged model...")
model.save_pretrained(
    str(MERGED_DIR),
    safe_serialization=True,
    max_shard_size="2GB",
)

tokenizer.save_pretrained(str(MERGED_DIR))

print("Saved merged model to:", MERGED_DIR)
for p in sorted(MERGED_DIR.iterdir()):
    if p.is_file():
        print(p.name, p.stat().st_size, "bytes")